# 01 Metrics, binary task and significance tests

Reads the predictions that the modelling notebooks exported and produces:

1. one table with every metric of every run (macro F1, MCC, AUC-PR, balanced accuracy,
   per class F1, and the per source majority vote),
2. the binary against three class comparison for **H2**,
3. confidence intervals with a **bootstrap over source addresses**,
4. significance tests for **H1** (distance), **H3** (packet against grouped split),
   **H4** (in domain against transfer) and the model comparison,
5. optionally the variability over the two hour blocks (leave one block out, retrains the
   random forest).

Why a bootstrap over sources and not over packets: the packets of one address are strongly
correlated, so treating them as independent samples gives intervals that are far too narrow
and p values that are far too small. Resampling whole addresses respects that structure.
McNemar is reported as well, because it is the textbook paired test, but it treats packets as
independent and is therefore optimistic.

Sections 1 to 4 need no training and take seconds to a few minutes.

In [ ]:
import sys, pathlib, importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
sys.path.append(str(pathlib.Path.cwd()))  # the notebooks live in code/analysis
import analysis_utils as au
importlib.reload(au)

RESULTS = pathlib.Path('results')
FIGURES = RESULTS / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)

EXPS = au.available_experiments()
print('experiments with exported artifacts:', EXPS)
missing = [e for e in au.EXPERIMENTS if e not in EXPS]
if missing:
    print('NOT YET RERUN (no artifacts):', missing)

## 1. All metrics of all runs

In [ ]:
m = au.load_all_metrics()
m['exp'] = m['experiment'].map(au.exp_name)
cols = ['exp', 'label', 'model', 'split', 'frame', 'macro_f1', 'mcc', 'auc_pr_macro', 'auc_pr_targets',
        'balanced_acc', 'acc', 'f1_0', 'f1_1', 'f1_2',
        'src_n', 'src_macro_f1', 'src_mcc', 'src_target_recall']
table = m[[c for c in cols if c in m.columns]].sort_values(['label', 'frame', 'model', 'split', 'exp'])
table.to_csv(RESULTS / 'metrics_all_runs.csv', index=False)
print('wrote', RESULTS / 'metrics_all_runs.csv')
table[table['model'] == 'Random Forest'].round(3)

### Random forest overview: packet level against per source majority vote

`src_*` columns: one decision per address (majority of its packet predictions). This is the
deployment view from the design protocol table. `src_target_recall` is the share of group
addresses that are assigned to the correct group.

In [ ]:
rf = table[table['model'] == 'Random Forest']
overview = rf.pivot_table(index=['label', 'exp'], columns=['frame', 'split'],
                          values=['macro_f1', 'mcc', 'auc_pr_macro', 'src_macro_f1'])
overview.round(3)

## 2. Binary against three class task (H2)

H2 expects that **detection** (background against any group) stays strong while the three class
task mostly fails at **attribution** (group 1 against group 2).

Two views:
* the binary models themselves (label `binary`), and
* the three class random forest read as a detector: of all target packets, how many are
  detected as *some* group, how many get the correct group, how many are confused with the
  other group and how many are lost to the background.

In [ ]:
rows = []
for e in EXPS:
    for split in ['packet', 'grouped']:
        for frame in ['indomain', 'transfer']:
            p = au.load_predictions(e, 'multi', 'random_forest', split, frame, with_meta=False)
            t, y = p['y_true'].to_numpy(), p['y_pred'].to_numpy()
            tgt = t != 0
            n = max(tgt.sum(), 1)
            err = tgt & (y != t)
            confused = tgt & (y != 0) & (y != t)
            rows.append(dict(exp=au.exp_name(e), split=split, frame=frame,
                             target_packets=int(tgt.sum()),
                             detected_as_any_group=(tgt & (y != 0)).sum() / n,
                             correct_group=(tgt & (y == t)).sum() / n,
                             confused_with_other_group=confused.sum() / n,
                             lost_to_background=(tgt & (y == 0)).sum() / n,
                             group_confusion_share_of_errors=confused.sum() / max(err.sum(), 1)))
h2 = pd.DataFrame(rows)

binary = (m[(m['model'] == 'Random Forest') & (m['label'] == 'binary')]
          .set_index(['exp', 'split', 'frame'])[['macro_f1', 'mcc', 'f1_1']]
          .rename(columns={'macro_f1': 'binary_macro_f1', 'mcc': 'binary_mcc', 'f1_1': 'binary_f1_target'}))
multi = (m[(m['model'] == 'Random Forest') & (m['label'] == 'multi')]
         .set_index(['exp', 'split', 'frame'])[['macro_f1', 'mcc']]
         .rename(columns={'macro_f1': 'multi_macro_f1', 'mcc': 'multi_mcc'}))
h2 = h2.join(binary, on=['exp', 'split', 'frame']).join(multi, on=['exp', 'split', 'frame'])
h2.to_csv(RESULTS / 'h2_binary_vs_multi.csv', index=False)
h2.round(3)

In [ ]:
sub = h2[(h2['split'] == 'grouped') & (h2['frame'] == 'transfer')].set_index('exp')
ax = sub[['correct_group', 'confused_with_other_group', 'lost_to_background']].plot.bar(
    stacked=True, figsize=(9, 4), color=['tab:green', 'tab:orange', 'tab:grey'])
ax.set_ylabel('share of target packets')
ax.set_title('Three class random forest on the university capture (grouped split)')
ax.legend(loc='upper right', fontsize=8)
plt.xticks(rotation=15, ha='right'); plt.tight_layout()
plt.savefig(FIGURES / 'fig_h2_target_packet_outcome.pdf'); plt.show()

## 3. Confidence intervals (bootstrap over source addresses)

`MODELS` controls which learners are included. On the university capture all runs of one
experiment and label share the same packets, so they are bootstrapped together (paired). The
in domain test sets differ between the packet and the grouped split, so they are bootstrapped
separately.

In [ ]:
N_BOOT = 1000
LABELS = ['multi', 'binary']
MODELS = ['random_forest', 'multi_layer_perceptron', 'stochastic_gradient_descent']
SPLITS = ['packet', 'grouped']
METRICS = ['macro_f1', 'mcc']

boot = {}      # (exp, label, frame, metric) -> DataFrame with one column per 'model|split'
ci_rows = []
for ei, e in enumerate(EXPS):
    for label in LABELS:
        # transfer: paired over all runs
        preds = {f'{mo}|{sp}': au.load_predictions(e, label, mo, sp, 'transfer') for mo in MODELS for sp in SPLITS}
        first = next(iter(preds.values()))
        for metric in METRICS:
            b = au.cluster_bootstrap(first['source'], first['y_true'], {k: v['y_pred'] for k, v in preds.items()},
                                     metric, N_BOOT, seed=100 + ei)
            boot[(e, label, 'transfer', metric)] = b
        # in domain: one bootstrap per run
        for mo in MODELS:
            for sp in SPLITS:
                p = au.load_predictions(e, label, mo, sp, 'indomain')
                for metric in METRICS:
                    b = au.cluster_bootstrap(p['source'], p['y_true'], {f'{mo}|{sp}': p['y_pred']},
                                             metric, N_BOOT, seed=200 + ei)
                    boot.setdefault((e, label, 'indomain', metric), pd.DataFrame({'replicate': b['replicate']}))
                    boot[(e, label, 'indomain', metric)][f'{mo}|{sp}'] = b[f'{mo}|{sp}']

for (e, label, frame, metric), b in boot.items():
    for run in [c for c in b.columns if c != 'replicate']:
        mo, sp = run.split('|')
        lo, hi = au.ci(b[run])
        ci_rows.append(dict(exp=au.exp_name(e), label=label, frame=frame, metric=metric, model=mo, split=sp,
                            boot_mean=b[run].mean(), ci_low=lo, ci_high=hi))
cis = pd.DataFrame(ci_rows)
point = (m.assign(model=m['model'].map(au.slug))
          .melt(id_vars=['exp', 'label', 'model', 'split', 'frame'], value_vars=METRICS,
                var_name='metric', value_name='point'))
cis = cis.merge(point, on=['exp', 'label', 'model', 'split', 'frame', 'metric'], how='left')
cis.to_csv(RESULTS / 'confidence_intervals.csv', index=False)
cis[(cis['model'] == 'random_forest') & (cis['metric'] == 'macro_f1')].round(3)

In [ ]:
# Transfer macro F1 of the random forest with 95 % intervals
sub = cis[(cis['model'] == 'random_forest') & (cis['metric'] == 'macro_f1') & (cis['frame'] == 'transfer')]
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, label in zip(axes, LABELS):
    s = sub[sub['label'] == label]
    for k, sp in enumerate(SPLITS):
        q = s[s['split'] == sp]
        x = np.arange(len(q)) + (k - 0.5) * 0.3
        ax.errorbar(x, q['point'], yerr=[q['point'] - q['ci_low'], q['ci_high'] - q['point']],
                    fmt='o', capsize=3, label=sp)
    ax.set_xticks(np.arange(len(q))); ax.set_xticklabels(q['exp'], rotation=20, ha='right')
    ax.axhline(1 / 3 if label == 'multi' else 0.5, ls='--', lw=1, color='grey')
    ax.set_title(f'{label} task, university capture'); ax.set_ylim(0, 1)
axes[0].set_ylabel('macro F1 (95 % source bootstrap)'); axes[0].legend()
plt.tight_layout(); plt.savefig(FIGURES / 'fig_ci_transfer_macro_f1.pdf'); plt.show()

## 4. Significance tests

* **H3** packet against grouped split: on the university capture the two runs are paired (same
  packets): paired source bootstrap and McNemar. In domain the test sets differ: unpaired.
* **H4** in domain against transfer: unpaired (different test sets).
* **H1** same against different distance (E1/E2, E3/E4, E5/E6): unpaired.
* **Models**: random forest against MLP and SGD on the university capture: paired.

All p values of one hypothesis family are Holm corrected. Primary metric: macro F1.

In [ ]:
METRIC = 'macro_f1'
tests = []

def add(hyp, exp, label, frame, a, b, res):
    tests.append(dict(hypothesis=hyp, exp=exp, label=label, frame=frame, a=a, b=b, **res))

for e in EXPS:
    for label in LABELS:
        bt = boot[(e, label, 'transfer', METRIC)]
        bi = boot[(e, label, 'indomain', METRIC)]
        # H3, transfer, paired
        res = au.paired_bootstrap_test(bt, 'random_forest|packet', 'random_forest|grouped')
        pa = au.load_predictions(e, label, 'random_forest', 'packet', 'transfer', with_meta=False)
        pg = au.load_predictions(e, label, 'random_forest', 'grouped', 'transfer', with_meta=False)
        res.update(au.mcnemar(pa['y_true'], pa['y_pred'], pg['y_pred']))
        add('H3', au.exp_name(e), label, 'transfer', 'packet', 'grouped', res)
        # H3, in domain, unpaired
        add('H3', au.exp_name(e), label, 'indomain', 'packet', 'grouped',
            au.unpaired_bootstrap_test(bi['random_forest|packet'], bi['random_forest|grouped']))
        # H4, grouped split
        add('H4', au.exp_name(e), label, 'indomain-transfer', 'indomain', 'transfer',
            au.unpaired_bootstrap_test(bi['random_forest|grouped'], bt['random_forest|grouped']))
        # models, transfer, paired
        for other in [mo for mo in MODELS if mo != 'random_forest']:
            for sp in SPLITS:
                res = au.paired_bootstrap_test(bt, f'random_forest|{sp}', f'{other}|{sp}')
                add('models', au.exp_name(e), label, f'transfer/{sp}', 'random_forest', other, res)

# H1: same against different distance
for same, diff in au.DISTANCE_PAIRS:
    if same not in EXPS or diff not in EXPS:
        continue
    for label in LABELS:
        for frame in ['transfer', 'indomain']:
            for sp in SPLITS:
                add('H1', f'{au.exp_name(same)} / {au.exp_name(diff)}', label, f'{frame}/{sp}', 'same', 'diff',
                    au.unpaired_bootstrap_test(boot[(same, label, frame, METRIC)][f'random_forest|{sp}'],
                                               boot[(diff, label, frame, METRIC)][f'random_forest|{sp}']))

tests = pd.DataFrame(tests)
tests['p_holm'] = np.nan
for hyp, idx in tests.groupby('hypothesis').groups.items():
    tests.loc[idx, 'p_holm'] = au.holm(tests.loc[idx, 'p_boot'])
tests['significant_5pct'] = tests['p_holm'] < 0.05
tests.to_csv(RESULTS / 'significance_tests.csv', index=False)
tests.round(4)

## 5. Variability over the two hour blocks (optional, retrains)

Leave one block out with the random forest: train on all but one two hour block, test on the
left out block (a later or earlier session of the same campaign) and on the university
capture. This gives one score per block, so the spread between blocks can be reported (the
design argues for five independent recording units).

Runtime: one random forest fit per block, label and experiment (about 1 to 3 minutes each).

In [ ]:
RUN_BLOCK_CV = True
BLOCK_LABELS = ['group_label_multi', 'group_label']

if RUN_BLOCK_CV:
    from sklearn.metrics import f1_score, matthews_corrcoef
    block_rows = []
    for e in EXPS:
        tr = au.load_frame(e, 'train')
        ev = au.load_frame(e, 'eval')
        feats = au.feature_columns(tr)
        for label in BLOCK_LABELS:
            for b in sorted(tr['block'].unique()):
                held = tr['block'] == b
                model = au.make_random_forest().fit(tr.loc[~held, feats], tr.loc[~held, label])
                for frame, df in [('left out block', tr[held]), ('transfer', ev)]:
                    y = model.predict(df[feats])
                    block_rows.append(dict(exp=au.exp_name(e), label=label, block=int(b), frame=frame,
                                           macro_f1=f1_score(df[label], y, average='macro', zero_division=0),
                                           mcc=matthews_corrcoef(df[label], y)))
                print(e, label, 'block', b, 'done')
            del model
    blocks = pd.DataFrame(block_rows)
    blocks.to_csv(RESULTS / 'block_variability.csv', index=False)
    display(blocks.groupby(['label', 'frame', 'exp'])[['macro_f1', 'mcc']].agg(['mean', 'std', 'min', 'max']).round(3))